# Two-observer spacetime-diagram tool

A small, Colab-friendly tool for constructing diagrams in **$(ct,x)$ coordinates**. Set the boost $\beta=u/c$ below; the $S'$ axes and their tick positions are then obtained from the Lorentz transformation.

Run the setup cell, edit the configuration and drawing cells, then export an SVG or PDF.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# A dimensionless speed beta=u/c must satisfy |beta|<1.
def gamma(beta):
    if abs(beta) >= 1:
        raise ValueError('beta must satisfy |beta| < 1.')
    return 1 / np.sqrt(1 - beta**2)

def two_observer_grid(beta=0.6, limit=6, tick_step=1, labels=True, grid=True):
    """Return fig, ax for a correctly scaled (ct,x) two-observer diagram.

    Matplotlib coordinates are (x, ct).  S' moves at beta along +x.
    The ct' axis is x'=0; the x' axis is ct'=0.
    """
    g = gamma(beta)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$ct$', rotation=0, labelpad=12)
    ax.axhline(0, color='black', lw=1.3)
    ax.axvline(0, color='black', lw=1.3)
    if grid:
        ax.set_xticks(np.arange(-limit, limit + tick_step, tick_step))
        ax.set_yticks(np.arange(-limit, limit + tick_step, tick_step))
        ax.grid(True, color='0.88', zorder=0)

    # S' axes: x=beta ct and ct=beta x.
    z = np.linspace(-limit, limit, 400)
    ax.plot(beta*z, z, color='tab:blue', lw=1.8)       # ct' axis
    ax.plot(z, beta*z, color='tab:blue', lw=1.8)       # x' axis

    # A ct'=n tick is (ct,x)=(gamma n, gamma beta n);
    # an x'=n tick is (ct,x)=(gamma beta n, gamma n).
    nmax = int(limit / (g * tick_step))
    for n in range(-nmax, nmax + 1):
        if n == 0:
            continue
        # (x,ct) positions, followed by short perpendicular tick marks
        x_ct, y_ct = g*beta*n*tick_step, g*n*tick_step
        x_x, y_x = g*n*tick_step, g*beta*n*tick_step
        normal_ct = np.array([1, -beta]); normal_ct /= np.linalg.norm(normal_ct)
        normal_x = np.array([-beta, 1]); normal_x /= np.linalg.norm(normal_x)
        for x, y, normal in [(x_ct, y_ct, normal_ct), (x_x, y_x, normal_x)]:
            d = 0.11 * normal
            ax.plot([x-d[0], x+d[0]], [y-d[1], y+d[1]], color='tab:blue', lw=1)

    if labels:
        ax.text(limit*0.95, beta*limit*0.95, "$x'$", color='tab:blue', ha='right', va='bottom')
        ax.text(beta*limit*0.95, limit*0.95, "$ct'$", color='tab:blue', ha='left', va='top')
    return fig, ax

def add_event(ax, x, ct, label=None, color='crimson', dx=0.12, dy=0.12):
    ax.plot(x, ct, 'o', color=color, ms=5, zorder=5)
    if label:
        ax.text(x+dx, ct+dy, label, color=color)

def add_worldline(ax, x0, ct0, beta, ct_range=(-6, 6), label=None, **style):
    """Worldline x=x0+beta(ct-ct0); beta=0 gives a stationary object."""
    ct = np.array(ct_range)
    x = x0 + beta*(ct-ct0)
    line, = ax.plot(x, ct, **({'lw': 2} | style))
    if label:
        ax.text(x[-1]+0.08, ct[-1], label, color=line.get_color())
    return line

def add_light_ray(ax, x0, ct0, direction=1, ct_range=(-6, 6), label=None, **style):
    """A light ray has beta=+1 (right) or -1 (left)."""
    return add_worldline(ax, x0, ct0, direction, ct_range, label, ls='--', **style)

def add_train_region(ax, rear_x0, length, beta, ct0=0, ct1=5, label=None, color='tab:orange', alpha=0.18):
    """Shade the world region of a train moving at beta."""
    front_x0 = rear_x0 + length
    points = [(rear_x0 + beta*(ct0-ct0), ct0),
              (front_x0 + beta*(ct0-ct0), ct0),
              (front_x0 + beta*(ct1-ct0), ct1),
              (rear_x0 + beta*(ct1-ct0), ct1)]
    ax.add_patch(Polygon(points, closed=True, facecolor=color, edgecolor=color, alpha=alpha))
    add_worldline(ax, rear_x0, ct0, beta, (ct0, ct1), color=color)
    add_worldline(ax, front_x0, ct0, beta, (ct0, ct1), color=color)
    if label:
        ax.text(np.mean([p[0] for p in points]), (ct0+ct1)/2, label, ha='center', color=color)


## Configuration
Change `beta`, the plotting range, and the tick separation. The blue ticks represent equal intervals of either $ct'$ or $x'$; they are not equally spaced in the unprimed coordinates.

In [ ]:
beta = 3/5       # speed u/c of S' relative to S
limit = 6        # extent of both x and ct axes
tick_step = 1    # one primed-coordinate unit per tick

fig, ax = two_observer_grid(beta, limit, tick_step)
ax.set_title(f'Two-observer diagram: $u/c={beta:g}$')
plt.show()

## Add events, worldlines, light rays, and a world region
The example below can be replaced freely. Coordinates supplied to the helper functions are always $(x,ct)$ in frame $S$. A worldline has `beta = dx/d(ct) = v/c`.

In [ ]:
# Start with a fresh grid.
fig, ax = two_observer_grid(beta=3/5, limit=6, tick_step=1)

# Events: add_event(ax, x, ct, label)
add_event(ax, -2, 1, '$A$')
add_event(ax,  2, 1, '$B$')

# Worldlines: x=x0+beta_line(ct-ct0)
add_worldline(ax, x0=-3, ct0=0, beta=0, ct_range=(-1, 5), label='platform clock', color='black')
add_worldline(ax, x0=-2, ct0=0, beta=3/5, ct_range=(-1, 6), label='traveller', color='tab:orange')

# Light rays (direction +1 is rightward, -1 is leftward).
add_light_ray(ax, x0=-2, ct0=1, direction=1, ct_range=(1, 5), label='light', color='crimson')

# A finite object is represented by the shaded region between its endpoint worldlines.
add_train_region(ax, rear_x0=0, length=2, beta=3/5, ct0=0, ct1=4, label='train')

ax.set_title('Example: events, worldlines, a light ray, and a train world region')
plt.show()

# Vector exports are best for lecture notes and slides.
# fig.savefig('two_observer_diagram.svg', bbox_inches='tight')
# fig.savefig('two_observer_diagram.pdf', bbox_inches='tight')

## Conventions and checks

- The horizontal coordinate is $x$ and the vertical coordinate is $ct$.
- $S'$ moves in the positive $x$ direction at $u=\beta c$.
- The $ct'$ axis is the worldline $x'=0$, i.e. $x=\beta ct$. The $x'$ axis is $ct'=0$, i.e. $ct=\beta x$.
- Tick positions use the inverse Lorentz transformation: a $ct'=L$ tick is $(ct,x)=(\gamma L,\gamma\beta L)$; an $x'=L$ tick is $(ct,x)=(\gamma\beta L,\gamma L)$.
- This notebook intentionally does not infer the physics of a diagram from its appearance. Specify the events and frame conditions first.